In [ ]:
# Notebook imports
# Generated from standard/third-party imports used throughout this notebook.
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import torch
import xarray as xr
from pathlib import Path


# NAC Scratch

Exploratory cells for NAC datasets: COCO release inspection, PHO/DTM metadata checks, and datamodule normalization diagnostics.


# NAC

## Look at NAC dataset

In [6]:
LFM_DIR = Path("/explore/nobackup/projects/lfm/")

In [7]:
nac_path = LFM_DIR / "processed_data/Lunar/data_release/NAC_craters_coco_release_final/"
contents = list(nac_path.glob("*"))
contents

[PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/DTM'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/PHO'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/splits'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/metadata.parquet'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/nac_handlabeled_annotations.json')]

In [3]:
pq_path = next(nac_path.glob("*.parquet"))
# pq_path = [p for p in [str(elem) for elem in contents] if ".parquet" in p][0]
df = pd.read_parquet(pq_path)
df.head()

,PRODUCT_ID,PHO_TILE,DTM_TILE,LTM_CODE,EMISSION_ANGLE,INCIDENCE_ANGLE,PHASE_ANGLE,SUB_SOLAR_GROUND_AZIMUTH,SUB_SOLAR_LATITUDE,SUB_SOLAR_LONGITUDE,UPPER_LEFT_LONGITUDE,LOWER_RIGHT_LONGITUDE,UPPER_LEFT_LATITUDE,LOWER_RIGHT_LATITUDE,CENTER_LONGITUDE,CENTER_LATITUDE,DATASET
0,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.831589,-11.823145,-0.457667,-0.449225,-11.827367,-0.453446,train
1,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.823145,-11.814701,-0.457667,-0.449225,-11.818923,-0.453446,train
2,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.814701,-11.806258,-0.457667,-0.449225,-11.810480,-0.453446,train
3,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.806258,-11.797814,-0.457667,-0.449225,-11.802036,-0.453446,train
4,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.831589,-11.823145,-0.466109,-0.457667,-11.827367,-0.461888,train


In [8]:
first_item = dict(df.iloc[0])
first_pho_tile = nac_path / first_item['PHO_TILE']
first_dtm_tile =  nac_path / first_item['DTM_TILE']
first_pho_tile, first_dtm_tile

(PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_PHO_r0_c0.nc'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_DTM_r0_c0.nc'))

In [11]:
pho_ds = xr.open_dataset(first_pho_tile)
print(pho_ds)

<xarray.Dataset> Size: 266kB
Dimensions:      (y: 256, x: 256)
Coordinates:
  * y            (y) float64 2kB -1.362e+04 -1.362e+04 ... -1.388e+04 -1.388e+04
  * x            (x) float64 2kB 5.099e+06 5.099e+06 ... 5.099e+06 5.099e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (y, x) float32 262kB ...
Attributes:
    crs:      PROJCS["Equirectangular_Moon",GEOGCS["GCS_Moon",DATUM["D_Moon",...


In [12]:
dtm_ds = xr.open_dataset(first_dtm_tile)
print(dtm_ds)

<xarray.Dataset> Size: 266kB
Dimensions:      (y: 256, x: 256)
Coordinates:
  * y            (y) float64 2kB -1.362e+04 -1.362e+04 ... -1.388e+04 -1.388e+04
  * x            (x) float64 2kB 5.099e+06 5.099e+06 ... 5.099e+06 5.099e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (y, x) float32 262kB ...
Attributes:
    crs:      PROJCS["Equirectangular_Moon",GEOGCS["GCS_Moon",DATUM["D_Moon",...


## Longer check for Codex

### First check

In [13]:

nac_path = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final")

print("Top level:")
for p in sorted(nac_path.iterdir()):
  print(p.name, "dir" if p.is_dir() else "file")

df = pd.read_parquet(nac_path / "metadata.parquet")
print("\nColumns:", list(df.columns))
print("\nSplit counts:")
print(df["DATASET"].value_counts(dropna=False))

print("\nPHO count:", len(list((nac_path / "PHO").glob("*.nc"))))
print("DTM count:", len(list((nac_path / "DTM").glob("*.nc"))))

print("\nSplits dir:")
for p in sorted((nac_path / "splits").glob("*")):
  print(p.name)

with open(nac_path / "nac_handlabeled_annotations.json") as f:
  coco = json.load(f)

print("\nCOCO keys:", coco.keys())
print("images:", len(coco.get("images", [])))
print("annotations:", len(coco.get("annotations", [])))
print("categories:", coco.get("categories", [])[:5])
print("\nFirst image:", coco.get("images", [None])[0])
print("\nFirst annotation:", coco.get("annotations", [None])[0])

Top level:
DTM dir
PHO dir
metadata.parquet file
nac_handlabeled_annotations.json file
splits dir

Columns: ['PRODUCT_ID', 'PHO_TILE', 'DTM_TILE', 'LTM_CODE', 'EMISSION_ANGLE', 'INCIDENCE_ANGLE', 'PHASE_ANGLE', 'SUB_SOLAR_GROUND_AZIMUTH', 'SUB_SOLAR_LATITUDE', 'SUB_SOLAR_LONGITUDE', 'UPPER_LEFT_LONGITUDE', 'LOWER_RIGHT_LONGITUDE', 'UPPER_LEFT_LATITUDE', 'LOWER_RIGHT_LATITUDE', 'CENTER_LONGITUDE', 'CENTER_LATITUDE', 'DATASET']

Split counts:
DATASET
train    645
test      62
val       59
Name: count, dtype: int64

PHO count: 766
DTM count: 766

Splits dir:
test.json
train.json
val.json

COCO keys: dict_keys(['info', 'images', 'categories', 'annotations'])
images: 766
annotations: 97104
categories: [{'id': 1, 'name': 'crater', 'supercategory': None}]

First image: {'id': 205, 'file_name': 'A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_PHO_r0_c0.nc', 'height': 256, 'width': 256, 'license': 0}

First annotation: {'id': 10329, 'image_id': 205, 'category_id': 1, 'segmentation': [[1, 237, 2, 238

### Second check

In [14]:

nac_path = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final")

for split in ["train", "val", "test"]:
  with open(nac_path / "splits" / f"{split}.json") as f:
      obj = json.load(f)
  print(split, type(obj))
  if isinstance(obj, dict):
      print(obj.keys())
      for k, v in obj.items():
          if isinstance(v, list):
              print(k, len(v), v[0] if v else None)
  elif isinstance(obj, list):
      print(len(obj), obj[0] if obj else None)
  print()

train <class 'dict'>
dict_keys(['info', 'images', 'categories', 'annotations'])
images 645 {'id': 205, 'file_name': 'A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_PHO_r0_c0.nc', 'height': 256, 'width': 256, 'license': 0}
categories 1 {'id': 1, 'name': 'crater', 'supercategory': None}
annotations 84536 {'id': 10329, 'image_id': 205, 'category_id': 1, 'segmentation': [[1, 237, 2, 238, 4, 239, 6, 241, 7, 242, 9, 243, 11, 244, 13, 244, 15, 245, 17, 246, 19, 246, 21, 247, 23, 247, 25, 248, 27, 248, 29, 248, 31, 248, 33, 248, 35, 248, 37, 248, 39, 248, 41, 247, 43, 247, 45, 246, 47, 246, 49, 245, 51, 244, 53, 244, 54, 243, 56, 242, 58, 241, 60, 239, 61, 238, 63, 237, 64, 236, 66, 234, 67, 233, 69, 231, 70, 230, 71, 228, 72, 226, 74, 225, 75, 223, 76, 221, 76, 219, 77, 217, 78, 215, 79, 213, 79, 211, 80, 209, 80, 207, 80, 205, 80, 203, 81, 201, 81, 199, 81, 197, 80, 195, 80, 193, 80, 191, 80, 189, 79, 187, 79, 185, 78, 183, 77, 181, 76, 179, 76, 178, 75, 176, 74, 174, 72, 172, 71, 171, 70, 169, 

## NAC Datamodule Normalization/Blur Diagnostic

This compares five NAC chips read directly from disk against the same samples as loaded by the semantic datamodule. It plots the direct chip, the datamodule tensor, the de-normalized datamodule tensor, and the absolute difference. If the de-normalized image matches the direct chip, then the apparent blur is likely display/normalization contrast rather than a spatial resampling problem in the datamodule.


In [ ]:

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scratch.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]
if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

from lfm.full_model.all_tasks.datamodules.datamodule_utils import read_image_file
from lfm.full_model.all_tasks.utils import load_terramind_pretraining_stats
from lfm.toy_model.sem_seg.lightning_wrappers.toy_sem_seg_datamodule import (
    LunarSemanticSegmentationSplitDataModule,
)
from lfm.toy_model.sem_seg.lightning_wrappers.toy_sem_seg_from_instance_datamodule import (
    ToySemSegFromInstanceDataModule,
)



In [ ]:
# Configure this block for the NAC dataset you want to inspect.
NAC_DIAG_DATA_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/256_256_inputs/nac_coco_inst_seg")
NAC_DIAG_SPLIT = "train"
NAC_DIAG_N_SAMPLES = 5
NAC_DIAG_LABEL_SOURCE = "instance"  # "instance" for .npz masks, "semantic" for .npy labels
NAC_DIAG_IMAGE_FILE_TYPE = ".npy"  # use ".tif" for IMP-style converted chips
NAC_DIAG_IMAGE_SUFFIX = "_input_nac_chip"
NAC_DIAG_LABEL_SUFFIX = "_label"
NAC_DIAG_BAND_FILTER = [0]
NAC_DIAG_TARGET_SIZE = (256, 256)
NAC_DIAG_SPATIAL_TRANSFORM = "crop"
NAC_DIAG_NORMALIZE_INPUTS = True
NAC_DIAG_NORMALIZATION_SOURCE = "pretrain"  # "pretrain" or "finetune"
NAC_DIAG_MODALITY_INFO = Path("/explore/nobackup/people/ajkerr1/Lunar_FM/sandy_graha_pretrain_dir/modality_info.yaml")

if NAC_DIAG_NORMALIZE_INPUTS and NAC_DIAG_NORMALIZATION_SOURCE == "pretrain":
    nac_diag_mean, nac_diag_std = load_terramind_pretraining_stats(
        NAC_DIAG_MODALITY_INFO,
        normalization_modality="nac",
        band_filter=NAC_DIAG_BAND_FILTER,
    )
else:
    nac_diag_mean, nac_diag_std = None, None

DatamoduleClass = (
    ToySemSegFromInstanceDataModule
    if NAC_DIAG_LABEL_SOURCE == "instance"
    else LunarSemanticSegmentationSplitDataModule
)

nac_diag_dm = DatamoduleClass(
    data_root=NAC_DIAG_DATA_ROOT,
    batch_size=NAC_DIAG_N_SAMPLES,
    num_workers=0,
    target_size=NAC_DIAG_TARGET_SIZE,
    spatial_transform=NAC_DIAG_SPATIAL_TRANSFORM,
    band_filter=NAC_DIAG_BAND_FILTER,
    normalize_inputs=NAC_DIAG_NORMALIZE_INPUTS,
    means=nac_diag_mean,
    stds=nac_diag_std,
    scale_inputs=NAC_DIAG_NORMALIZATION_SOURCE != "pretrain",
    max_train_samples=max(NAC_DIAG_N_SAMPLES, 10),
    max_val_samples=max(NAC_DIAG_N_SAMPLES, 10),
    max_test_samples=max(NAC_DIAG_N_SAMPLES, 10),
    image_file_type=NAC_DIAG_IMAGE_FILE_TYPE,
    image_suffix=NAC_DIAG_IMAGE_SUFFIX,
    label_suffix=NAC_DIAG_LABEL_SUFFIX,
)
nac_diag_dm.setup("fit")

nac_diag_dataset = getattr(nac_diag_dm, f"{NAC_DIAG_SPLIT}_dataset")
print(f"Diagnostic split: {NAC_DIAG_SPLIT}")
print(f"Samples available: {len(nac_diag_dataset)}")
print(f"Normalize inputs: {NAC_DIAG_NORMALIZE_INPUTS}")
print(f"Mean: {nac_diag_dm.mean}")
print(f"Std: {nac_diag_dm.std}")



In [ ]:
def _to_hwc(image):
    image = np.asarray(image)
    if image.ndim == 2:
        return image[:, :, None]
    if image.ndim == 3:
        if image.shape[0] <= image.shape[-1]:
            return np.moveaxis(image, 0, -1)
        return image
    raise ValueError(f"Expected 2D or 3D image, got {image.shape}")


def _center_crop_hwc(image, target_size):
    target_h, target_w = target_size
    height, width = image.shape[:2]
    top = (height - target_h) // 2
    left = (width - target_w) // 2
    return image[top:top + target_h, left:left + target_w, :]


def _minmax_per_band(image):
    image = image.astype("float32", copy=True)
    for band_idx in range(image.shape[-1]):
        band = image[:, :, band_idx]
        finite = np.isfinite(band)
        if not finite.any():
            continue
        band_min = np.nanmin(band[finite])
        band_max = np.nanmax(band[finite])
        if band_max > band_min:
            image[:, :, band_idx] = (band - band_min) / (band_max - band_min)
    return image


def _direct_preprocessed_band(image_path, dataset):
    raw = _to_hwc(read_image_file(Path(image_path))).astype("float32", copy=False)
    raw = raw[:, :, dataset.band_filter]
    if dataset.scale_inputs:
        raw = _minmax_per_band(raw)
    if dataset.spatial_transform == "crop":
        raw = _center_crop_hwc(raw, dataset.target_size)
    elif dataset.spatial_transform == "resize":
        raise NotImplementedError("Diagnostic currently expects crop for NAC chips.")
    return raw[:, :, 0]


def _read_file_nodata_value(image_path):
    image_path = Path(image_path)
    suffix = image_path.suffix.lower()
    if suffix in {".tif", ".tiff"}:
        with rasterio.open(image_path) as src:
            return src.nodata
    if suffix == ".nc":
        with xr.open_dataset(image_path) as ds:
            if NETCDF_VARIABLE in ds:
                attrs = ds[NETCDF_VARIABLE].attrs
            elif len(ds.data_vars) == 1:
                attrs = next(iter(ds.data_vars.values())).attrs
            else:
                attrs = {}
            return attrs.get("_FillValue") or attrs.get("missing_value")
    return None


def _nodata_mask(arr, nodata_value=None):
    arr = np.asarray(arr)
    mask = ~np.isfinite(arr)
    # Common lunar/GDAL fill values seen in this repo.
    mask |= arr <= -1e30
    if nodata_value is not None and np.isfinite(nodata_value):
        mask |= np.isclose(arr, nodata_value)
    return mask


def _nodata_stats(arr, nodata_value=None, prefix="nodata"):
    arr = np.asarray(arr)
    mask = _nodata_mask(arr, nodata_value)
    total = int(arr.size)
    count = int(mask.sum())
    return {
        f"{prefix}_count": count,
        f"{prefix}_pct": float(count / total * 100.0) if total else np.nan,
    }


def _raw_band_before_datamodule_preprocessing(image_path, dataset):
    raw = _to_hwc(read_image_file(Path(image_path))).astype("float32", copy=False)
    raw = raw[:, :, dataset.band_filter]
    if dataset.spatial_transform == "crop":
        raw = _center_crop_hwc(raw, dataset.target_size)
    elif dataset.spatial_transform == "resize":
        raise NotImplementedError("Diagnostic currently expects crop for NAC chips.")
    return raw[:, :, 0]


def _as_percentile_display(image, lower=2, upper=98):
    image = np.asarray(image, dtype="float64")
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        return image
    vmin, vmax = np.percentile(finite, [lower, upper])
    if vmax <= vmin:
        return image
    return np.clip((image - vmin) / (vmax - vmin), 0, 1)


def _denormalize_loaded_band(loaded_band, dataset):
    if not dataset.normalize_inputs:
        return loaded_band
    mean = float(dataset.mean[0])
    std = float(dataset.std[0])
    return loaded_band * std + mean


def _metrics(a, b):
    a = np.asarray(a, dtype="float64")
    b = np.asarray(b, dtype="float64")
    mask = np.isfinite(a) & np.isfinite(b)
    mask &= ~_nodata_mask(a)
    mask &= ~_nodata_mask(b)
    if not mask.any():
        return {"mae": np.nan, "rmse": np.nan, "corr": np.nan}
    diff = b[mask] - a[mask]
    corr = np.corrcoef(a[mask], b[mask])[0, 1] if diff.size > 1 else np.nan
    return {
        "mae": float(np.mean(np.abs(diff))),
        "rmse": float(np.sqrt(np.mean(diff ** 2))),
        "corr": float(corr),
    }



In [ ]:
n_samples = min(NAC_DIAG_N_SAMPLES, len(nac_diag_dataset))
fig, axes = plt.subplots(4, n_samples, figsize=(3.4 * n_samples, 12), constrained_layout=True)
if n_samples == 1:
    axes = axes[:, None]

rows = []
for col in range(n_samples):
    item = nac_diag_dataset[col]
    loaded_image = item[0]
    image_path = item[2]
    filename = Path(image_path).name

    nodata_value = _read_file_nodata_value(image_path)
    raw_band = _raw_band_before_datamodule_preprocessing(image_path, nac_diag_dataset)
    direct_band = _direct_preprocessed_band(image_path, nac_diag_dataset)
    loaded_band = loaded_image[0].detach().cpu().numpy()
    denorm_band = _denormalize_loaded_band(loaded_band, nac_diag_dataset)
    abs_diff = np.abs(denorm_band - direct_band)
    stats = _metrics(direct_band, denorm_band)
    nodata_stats = {
        "file_nodata_value": nodata_value,
        **_nodata_stats(raw_band, nodata_value, prefix="raw_crop_nodata"),
        **_nodata_stats(direct_band, None, prefix="direct_preprocessed_nodata"),
        **_nodata_stats(loaded_band, None, prefix="loaded_tensor_nodata"),
        **_nodata_stats(denorm_band, None, prefix="denorm_tensor_nodata"),
    }
    rows.append({"filename": filename, **stats, **nodata_stats})

    panels = [direct_band, loaded_band, denorm_band, abs_diff]
    titles = [
        f"direct raw/cropped\nraw nodata={nodata_stats['raw_crop_nodata_pct']:.2f}%",
        f"datamodule tensor\nnodata={nodata_stats['loaded_tensor_nodata_pct']:.2f}%",
        f"datamodule denorm\nnodata={nodata_stats['denorm_tensor_nodata_pct']:.2f}%",
        f"abs diff\nMAE={stats['mae']:.3g}",
    ]
    for row_idx, (panel, title) in enumerate(zip(panels, titles)):
        ax = axes[row_idx, col]
        if row_idx == 3:
            im = ax.imshow(panel, cmap="magma")
        else:
            im = ax.imshow(_as_percentile_display(panel), cmap="gray")
        ax.set_title(title if row_idx else f"{filename}\n{title}", fontsize=9)
        ax.axis("off")

fig.suptitle("NAC Direct Chip vs Datamodule Loaded Tensor", fontsize=16, fontweight="bold")
plt.show()

nac_diag_metrics = pd.DataFrame(rows)
nac_diag_metrics

